###### This workflow will evaluate an essay on different aspects such as Clarity of thought,Depth of analysis , language and style, and Overall coherence. Each aspect will be scored on a scale of 1 to 10, with 1 being the lowest and 10 being the highest. The final score will be an average of the individual scores.

In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [ ]:
model = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
essay = f"""
# Education Is Not Merely a Tool for Personal Development, but a Foundation for Social Transformation

Education is one of the most powerful instruments for developing individuals and transforming society. It provides knowledge, skills, and critical thinking abilities that help people make informed decisions. However, the true purpose of education extends beyond securing employment or achieving personal success. It also plays a vital role in reducing inequality, strengthening democracy, and promoting social progress.

At the individual level, education improves employment opportunities and economic independence. It enables people to understand their rights, develop problem-solving skills, and make better decisions. For economically disadvantaged individuals, education can provide a path out of poverty and improve their quality of life.

Education also promotes social equality. In India, social reformers such as Savitribai Phule and Jyotirao Phule recognized education as a means of challenging caste discrimination and gender inequality. Educating women, in particular, can improve family health, increase workforce participation, and strengthen their role in decision-making. However, unequal access to quality education continues to disadvantage children from poor and rural backgrounds.

Furthermore, education contributes to economic growth by developing a skilled workforce and encouraging innovation. India's achievements in information technology and space research demonstrate the importance of scientific knowledge and technical expertise. Nevertheless, degrees alone cannot guarantee employment. A mismatch between academic curricula and industry requirements can result in unemployment, highlighting the need for practical skills and vocational training.

Education is equally important for democracy. Informed citizens can understand constitutional values, evaluate government policies, and participate responsibly in public life. It can also promote tolerance and respect for cultural diversity. However, education does not automatically guarantee responsible citizenship; ethical reasoning and media literacy are equally necessary to combat misinformation and prejudice.

Despite its benefits, India's education system faces several challenges, including inadequate infrastructure, unequal digital access, teacher shortages, and an excessive focus on memorization. These problems require greater investment in public education, improved teacher training, inclusive learning environments, and curricula that encourage creativity and critical thinking. Digital technology can support these efforts, but it should complement rather than replace effective teaching.

In conclusion, education is not merely a means of personal advancement but a foundation for building an inclusive and progressive society. Its success should be measured not only by academic qualifications or employment outcomes but also by its ability to reduce inequalities, strengthen democratic values, and encourage social responsibility. When education combines knowledge with ethical understanding and equal opportunity, it becomes a powerful force for lasting social transformation.
"""

In [ ]:
# Defining the schema for Structured Output

class EvaluationSchema(BaseModel):

    feedback: str = Field(description="Detailed feedback for essay")
    score: int = Field(description="Score based on feedback between 0 and 10, Do not return decimal values.", ge=0, le=10)

In [ ]:
model_with_structured_output = model.with_structured_output(EvaluationSchema)

In [ ]:
# Defining state 

class UPSCState(TypedDict):

    essay: str
    clarity_of_thought_feedback: str
    depth_of_analysis_feedback: str
    language_feedback: str
    individual_scores: Annotated[list[int], operator.add]

    overall_feedback: str
    average_score: float


In [ ]:
# definig graph

graph = StateGraph(UPSCState)

In [ ]:
def evaluate_clarity_of_thought(state: UPSCState):
    prompt = f"Evaluate the clarity of thought of the following essay and score it out of 10 -> {state['essay']}"

    result = model_with_structured_output.invoke(prompt)

    return {"clarity_of_thought_feedback": result.feedback, "individual_scores": [result.score]}

In [ ]:
def evaluate_depth_of_analysis(state: UPSCState):
    prompt = f"Evaluate the depth of analysis of the following essay and score it out of 10 -> {state['essay']}"

    result = model_with_structured_output.invoke(prompt)

    return {"depth_of_analysis_feedback": result.feedback, "individual_scores": [result.score]}

In [ ]:
def evaluate_language(state: UPSCState):
    prompt = f"Evaluate the language quality of the following essay and score it out of 10 -> {state['essay']}"

    result = model_with_structured_output.invoke(prompt)

    return {"language_feedback": result.feedback, "individual_scores": [result.score]}

In [ ]:
def final_evaluation(state: UPSCState):
    prompt = f"Based on the following feedback , generate summarized feedback \n Language feedback -> {state['language_feedback']} \n Clarity of thought feedback -> {state['clarity_of_thought_feedback']} \n Depth of analysis feedback -> {state['depth_of_analysis_feedback']}"

    result = model.invoke(prompt).content

    # Calculating average feedback score

    avg_feedback_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {"overall_feedback": result, "average_score": avg_feedback_score}

In [ ]:
# defining nodes and edges 

graph.add_node("evaluate_clarity_of_thought",evaluate_clarity_of_thought)
graph.add_node("evaluate_depth_of_analysis",evaluate_depth_of_analysis)
graph.add_node("evaluate_language",evaluate_language)
graph.add_node("final_evaluation",final_evaluation)

graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_depth_of_analysis")
graph.add_edge(START, "evaluate_clarity_of_thought")

graph.add_edge("evaluate_language", "final_evaluation")
graph.add_edge("evaluate_depth_of_analysis", "final_evaluation")
graph.add_edge("evaluate_clarity_of_thought", "final_evaluation")

graph.add_edge("final_evaluation", END)

workflow = graph.compile()
workflow

In [ ]:
initial_state= {
    "essay": essay
}
result = workflow.invoke(initial_state)

In [ ]:
result

In [ ]:
print(result['individual_scores'])

In [ ]:
poor_essay = f"""
# Education is Important for Society

Education is very important in our life because without education people cannot do anything and education is the thing which make people successful. Nowadays everyone says education is important but many people are educated and still they are not doing good things so education is good but also maybe not good for everyone.

Education help people to get jobs and earn money. If a person study more then he can earn more money, but sometimes educated people are unemployed also. This means education is important for getting job but job is not always getting from education. Government should do something because many students study but they are not getting jobs and this is a very big problem for India.

Education also helps society to become better. In old times people was not educated and many bad things was happening. Now people are educated but discrimination and corruption still happens. So education can remove these problems but these problems are also not removed even after education. This is why education is important and society need education to improve itself.

Girls education is also important because girls are also human and they should get education like boys. If girls study then their family will become educated and country will develop. But in some places girls are not allowed to study because of family problems and other reasons. Government should provide education to all girls because education is their right and everyone should understand this thing.

Education is also important for economy. When people study they can make new things and companies can earn more profit. India has many educated people but still poverty is there. This shows education can reduce poverty but poverty is not going away. If government gives education to all people then India will become developed country very soon because education solves many problems.

But education system has many problems. Schools are not good in some areas and teachers are not teaching properly. Students only study for marks and exams and they don't learn real things. Technology is also there but not everyone can use technology. These things are bad and government must improve education system. More schools should be build and teachers should teach students better so that education can become good.

In conclusion, education is very important for individuals and society because it makes people successful and society developed. But education is not solving all problems even though it is important. Government should provide education and people should study because without education country cannot progress. Therefore education is the most important thing and everyone should understand the importance of education.
"""

In [ ]:
initial_state = {
    "essay" : poor_essay
}

poor_result = workflow.invoke(initial_state)
poor_result